# Ephemeral RayJob — distributed compute

Same ephemeral pattern as Topic 2: `RayJob` + `ManagedClusterConfig`.

Runs `distributed_stats.py` with `@ray.remote` tasks across workers.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "extras/python"))
from workshop_common import (
    LOCAL_QUEUE,
    NAMESPACE,
    RayJob,
    cpu_cluster_config,
    login,
    print_job_logs,
    runtime_env_for_script,
    submit_rayjob,
    wait_for_rayjob,
)

In [ ]:
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN".strip()

auth = login(OPENSHIFT_TOKEN, OPENSHIFT_SERVER)
print(f"Namespace: {NAMESPACE}, Local queue: {LOCAL_QUEUE}")

from workshop_common import show_local_queues
show_local_queues(NAMESPACE)

In [ ]:
job = RayJob(
    job_name="ray-workshop-distributed-stats",
    entrypoint="python extras/scripts/distributed_stats.py",
    cluster_config=cpu_cluster_config(num_workers=2),
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    runtime_env=runtime_env_for_script(
        env_vars={
            "INPUT_PATH": "extras/data/iris.csv",
            "NUM_PARTITIONS": "4",
        },
    ),
    ttl_seconds_after_finished=600,
)
submit_rayjob(job)

In [ ]:
wait_for_rayjob(job)
print_job_logs(job)

## Check results

```sh
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=100 | tail -30
```

Expect JSON partition summaries and `Aggregated row count: 30`.